# 01 — Dataset Preparation (Kaggle / Colab / Local)

Load ViMD parquet → validate → extract features → split → export HuggingFace Dataset.

**Kaggle**: Data đọc từ `/kaggle/input/<slug>/ViMD/data/` — Output vào `/kaggle/working/`

**Checkpoint**: Skip nếu dataset đã export (`vietspeech-vimd/` đã có).

In [ ]:
# ── Cell 0: Environment + Paths ───────────────────────────────────────────────
import sys, os, gc

# Kaggle: thêm notebook dir vào sys.path để import _env
if os.path.exists('/kaggle/working/project/training/notebooks'):
    sys.path.insert(0, '/kaggle/working/project/training/notebooks')
elif os.path.exists('/kaggle/working/project'):
    sys.path.insert(0, '/kaggle/working/project')

from _env import *
from pathlib import Path
import numpy as np

print(f'ENV            : {ENV_NAME}')
print(f'VIMD_PATH      : {VIMD_PATH}')
print(f'DATASET_EXPORT : {DATASET_EXPORT}')

# ── Checkpoint skip ───────────────────────────────────────────────────────────
CKPT_DONE = os.path.exists(FINAL_DATASET)
if CKPT_DONE:
    print(f'\n✅ CHECKPOINT: Dataset đã export → {FINAL_DATASET}')
    print('   Bỏ qua processing — load từ disk')
else:
    print('\n⏳ Chưa có dataset — chạy toàn bộ pipeline...')

In [ ]:
# ── Cell 1: Dependencies ──────────────────────────────────────────────────────
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'soundfile', 'librosa', 'datasets', 'pyarrow', 'psutil'], check=False)
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Scan ViMD parquet shards ──────────────────────────────────────────
if CKPT_DONE:
    print('⏭️  Skip')
else:
    import glob, io
    import pandas as pd
    import soundfile as sf
    import pyarrow.parquet as pq

    parquet_files = sorted(glob.glob(os.path.join(VIMD_PATH, '*.parquet')))
    if not parquet_files:
        parquet_files = sorted(glob.glob(
            os.path.join(VIMD_PATH, '**', '*.parquet'), recursive=True))

    if not parquet_files:
        print(f'❌ Không tìm thấy .parquet trong: {VIMD_PATH}')
        if IS_KAGGLE:
            print(f'   Input datasets đang mount: {os.listdir("/kaggle/input")}')
            print(f'   Hãy add dataset "{VIMD_SLUG}" vào notebook')
        raise RuntimeError('Không tìm thấy parquet.')

    print(f'✓ {len(parquet_files)} parquet shards')
    schema_names = pq.read_schema(parquet_files[0]).names
    print(f'  Columns: {schema_names}')
    has_audio_col = 'audio' in schema_names
    print(f'  Has audio col: {has_audio_col}')

In [ ]:
# ── Cell 3: Build metadata (shard-by-shard) ───────────────────────────────────
if CKPT_DONE:
    print('⏭️  Skip')
else:
    BATCH_SHARDS = 10

    def get_audio_length(af) -> float:
        try:
            if isinstance(af, dict):
                arr = af.get('array')
                if arr is not None:
                    return len(arr) / af.get('sampling_rate', 16000)
                b = af.get('bytes')
            elif isinstance(af, bytes):
                b = af
            else:
                return 0.0
            if b:
                with sf.SoundFile(io.BytesIO(b)) as s:
                    return len(s) / s.samplerate
        except Exception:
            pass
        return 0.0

    all_meta = []
    for i in range(0, len(parquet_files), BATCH_SHARDS):
        batch_dfs = []
        for fp in parquet_files[i:i+BATCH_SHARDS]:
            try:
                df_s = pd.read_parquet(fp)
                if 'audio' in df_s.columns:
                    df_s['length'] = df_s['audio'].apply(get_audio_length)
                    df_s.drop(columns=['audio'], inplace=True)
                elif 'length' not in df_s.columns:
                    df_s['length'] = 0.0
                for col, default in [('province','unknown'),('ethnicity','viet'),
                                     ('language','vi'),('region','')]:
                    if col not in df_s.columns:
                        df_s[col] = df_s.get('province_name', default) if col == 'province' else default
                df_s['_shard'] = os.path.basename(fp)
                batch_dfs.append(df_s)
            except Exception as e:
                print(f'  ⚠️  {os.path.basename(fp)}: {e}')
        if batch_dfs:
            all_meta.append(pd.concat(batch_dfs, ignore_index=True))
        del batch_dfs; gc.collect()
        end = min(i + BATCH_SHARDS, len(parquet_files))
        print(f'Processed {i+1}–{end}/{len(parquet_files)}')

    df_all = pd.concat(all_meta, ignore_index=True)
    del all_meta; gc.collect()
    print(f'\n✓ Loaded {len(df_all):,} utterances')

In [ ]:
# ── Cell 4: Validate ──────────────────────────────────────────────────────────
if CKPT_DONE:
    print('⏭️  Skip')
else:
    mask = (df_all.get('length', pd.Series([0]*len(df_all))) >= 0.5) & \
           (df_all.get('length', pd.Series([30]*len(df_all))) <= 30.0)
    df_valid = df_all[mask].copy() if df_all.get('length', pd.Series([1])).max() > 0 else df_all.copy()
    del df_all; gc.collect()
    text_col = 'text' if 'text' in df_valid.columns else 'transcript'
    if text_col == 'transcript':
        df_valid = df_valid.rename(columns={'transcript': 'text'})
    df_valid = df_valid[df_valid['text'].notna() & (df_valid['text'].str.strip() != '')]
    print(f'Valid: {len(df_valid):,} samples')

In [ ]:
# ── Cell 5: Giải nén audio → /kaggle/working/storage/datasets/.../audio/ ──────
# Kaggle: audio lưu trong /kaggle/working → persist khi Save Version
if CKPT_DONE:
    print('⏭️  Skip')
else:
    from tqdm.auto import tqdm

    Path(AUDIO_DIR).mkdir(parents=True, exist_ok=True)
    existing_audio = set(os.listdir(AUDIO_DIR))
    print(f'AUDIO_DIR: {AUDIO_DIR} ({len(existing_audio):,} files đã có)')

    if 'filename' not in df_valid.columns:
        print('⚠️  Không có cột filename — skip')
    else:
        needed = set(df_valid['filename']) - existing_audio
        print(f'Cần extract: {len(needed):,}')
        if needed:
            fn_to_shard = {}
            for fp in parquet_files:
                try:
                    tmp = pd.read_parquet(fp, columns=['filename'])
                    for fn in tmp['filename']: fn_to_shard[fn] = fp
                    del tmp
                except Exception: pass
            gc.collect()

            shard_to_fns = {}
            for fn in needed:
                sp = fn_to_shard.get(fn)
                if sp: shard_to_fns.setdefault(sp, []).append(fn)

            extracted = 0
            for shard_path, fns in tqdm(shard_to_fns.items(), desc='Extracting'):
                try:
                    import soundfile as sf
                    df_s = pd.read_parquet(shard_path)
                    df_s = df_s[df_s['filename'].isin(set(fns))]
                    if 'audio' not in df_s.columns: continue
                    for _, row in df_s.iterrows():
                        out = os.path.join(AUDIO_DIR, row['filename'])
                        if os.path.exists(out): continue
                        af = row['audio']
                        try:
                            if isinstance(af, dict):
                                if 'array' in af and af['array'] is not None:
                                    sf.write(out, np.array(af['array'], dtype='float32'),
                                             int(af.get('sampling_rate', 16000)))
                                elif 'bytes' in af and af['bytes']:
                                    open(out, 'wb').write(af['bytes'])
                            elif isinstance(af, bytes):
                                open(out, 'wb').write(af)
                            extracted += 1
                        except Exception: pass
                    del df_s; gc.collect()
                except Exception as e:
                    print(f'  ⚠️  {os.path.basename(shard_path)}: {e}')
            print(f'✓ Extracted {extracted:,} → {AUDIO_DIR}')

In [ ]:
# ── Cell 6: Feature Extraction (checkpoint per batch) ─────────────────────────
if CKPT_DONE:
    print('⏭️  Skip')
else:
    import librosa, glob as _glob
    from tqdm.auto import tqdm

    existing_feat = sorted(_glob.glob(os.path.join(FEATURES_DIR, 'feat_batch_*.parquet')))
    if existing_feat:
        print(f'✅ CHECKPOINT: {len(existing_feat)} feature batches đã có')
    else:
        audio_on_disk = set(os.listdir(AUDIO_DIR)) if os.path.isdir(AUDIO_DIR) else set()
        df_wa = df_valid[df_valid['filename'].isin(audio_on_disk)].copy()
        print(f'Samples with audio: {len(df_wa):,}')

        for bi, df_b in enumerate([df_wa.iloc[i:i+1000] for i in range(0, len(df_wa), 1000)]):
            out_p = os.path.join(FEATURES_DIR, f'feat_batch_{bi:03d}.parquet')
            if os.path.exists(out_p):
                print(f'  Batch {bi}: exists')
                continue
            records = []
            for _, row in tqdm(df_b.iterrows(), total=len(df_b),
                               desc=f'Feat {bi+1}', leave=False):
                try:
                    arr, sr = librosa.load(os.path.join(AUDIO_DIR, row['filename']),
                                           sr=16000, mono=True, duration=30.0)
                    records.append({
                        'filename'   : row['filename'],
                        'text'       : str(row.get('text', '')),
                        'speaker'    : str(row.get('speakerID', row.get('speaker', ''))),
                        'gender'     : int(row.get('gender', -1)),
                        'province'   : str(row.get('province', '')),
                        'region'     : str(row.get('region', '')),
                        'ethnicity'  : str(row.get('ethnicity', 'viet')),
                        'language'   : str(row.get('language', 'vi')),
                        'duration'   : float(len(arr)/sr),
                        'energy_mean': float(np.sqrt(np.mean(arr**2))),
                    })
                except Exception: pass
            if records:
                pd.DataFrame(records).to_parquet(out_p, index=False)
                print(f'  Batch {bi+1}: {len(records):,} → {os.path.basename(out_p)}')
            gc.collect()
        print('\n✓ Feature extraction done')

In [ ]:
# ── Cell 7: Split + Export HuggingFace Dataset ────────────────────────────────
if CKPT_DONE:
    print('⏭️  Load từ disk')
    from datasets import load_from_disk
    ds = load_from_disk(FINAL_DATASET)
    print(ds)
else:
    import glob as _glob
    from datasets import Dataset, DatasetDict

    feat_files = sorted(_glob.glob(os.path.join(FEATURES_DIR, 'feat_batch_*.parquet')))
    if not feat_files:
        raise RuntimeError('Không có feature parquet. Chạy Cell 6.')

    df_feat = pd.concat([pd.read_parquet(p) for p in feat_files], ignore_index=True)
    aug = df_feat.copy()
    aug['is_augmented'] = True
    df_feat['is_augmented'] = False
    df_combined = pd.concat([df_feat, aug], ignore_index=True)
    del aug, df_feat; gc.collect()

    spk = 'speaker' if 'speaker' in df_combined.columns else 'speakerID'
    speakers = df_combined[spk].unique()
    rng = np.random.default_rng(42)
    rng.shuffle(speakers)
    n = len(speakers)
    test_s  = set(speakers[:max(1,int(n*.10))])
    val_s   = set(speakers[max(1,int(n*.10)):max(2,int(n*.20))])
    train_s = set(speakers[max(2,int(n*.20)):])

    Path(FINAL_DATASET).mkdir(parents=True, exist_ok=True)
    DatasetDict({
        'train': Dataset.from_pandas(df_combined[df_combined[spk].isin(train_s)].reset_index(drop=True), preserve_index=False),
        'valid': Dataset.from_pandas(df_combined[df_combined[spk].isin(val_s)].reset_index(drop=True),  preserve_index=False),
        'test' : Dataset.from_pandas(df_combined[df_combined[spk].isin(test_s)].reset_index(drop=True), preserve_index=False),
    }).save_to_disk(FINAL_DATASET)
    del df_combined; gc.collect()

    print(f'\n✅ Dataset saved → {FINAL_DATASET}')

if IS_KAGGLE:
    print('\n⚠️  Kaggle: Nhấn "Save Version" để persist output!')